# Rung 5 — The joint posterior

**Question.** Rung 4 gave intervals and said you could stop. What is worth climbing for?

Not tighter error bars. Two things the previous rung structurally cannot do:

- **Use what is already known.** Serial-interval studies constrain `1/gamma`; serosurveys
  constrain `rho`. A likelihood has no channel for any of that.
- **Answer a question in more than one dimension.** Rung 3 found `beta` and `gamma` to be
  strongly correlated. Pushing two separate intervals through a policy question
  double-counts uncertainty the data already resolved.

This notebook needs the optional `bayes` dependency group: `uv sync --group bayes`.

> **A note on the sampler.** The plan called for NUTS. NUTS needs the gradient of the
> log-posterior, and this SIR is integrated by `scipy`, which automatic differentiation
> cannot see through. Getting gradients would mean a second implementation of the same
> dynamics in a differentiable framework — precisely the drift the model-agnostic design
> exists to avoid. So the likelihood is sampled as a black box. That is a real cost, and
> the second half of this notebook shows it is not a free one.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from inferladder import default_sir_dataset, default_sir_model
from inferladder.diagnostics import observed_fisher
from inferladder.inference import fit, laplace_intervals, negative_log_likelihood
from inferladder.inference.bayes import (
    Priors,
    check_convergence,
    posterior_samples,
    sample_posterior,
)

dataset = default_sir_dataset()
model = default_sir_model()
truth = dataset.truth
Priors()

Those priors are claims, so they are written down rather than buried. `rho` carries the
one that does real work: `Beta(2, 4)` says most reporting fractions sit well below one
half, which is the kind of knowledge the likelihood alone cannot supply.

## Sampling

In [ ]:
idata = sample_posterior(model, dataset.y, dataset.t, draws=1000, tune=1000, seed=1)

report = check_convergence(idata, var_names=["beta", "gamma", "rho", "k", "R0"])
print(f"converged     : {report.converged}")
print(f"max R-hat     : {report.max_r_hat:.4f}")
print(f"min bulk ESS  : {report.min_ess_bulk:.0f}")

Convergence is checked before the numbers are read, not after. On a correlated posterior
a stuck chain does not produce noisy answers — it produces *narrow* ones, so the failure
mode is overconfidence.

In [ ]:
samples = posterior_samples(idata, model.param_names)
r0 = samples[:, 0] / samples[:, 1]

print(f"{'param':6} {'post. mean':>11} {'95% credible':>22} {'truth':>9}  covers")
for index, name in enumerate(model.param_names):
    low, high = np.percentile(samples[:, index], [2.5, 97.5])
    covers = "yes" if low <= truth[name] <= high else "NO"
    print(
        f"{name:6} {samples[:, index].mean():11.4f} {f'[{low:.4f}, {high:.4f}]':>22} "
        f"{truth[name]:9.4f}  {covers}"
    )

low, high = np.percentile(r0, [2.5, 97.5])
print(
    f"{'R0':6} {r0.mean():11.4f} {f'[{low:.4f}, {high:.4f}]':>22} "
    f"{truth['beta'] / truth['gamma']:9.4f}  {'yes' if low <= 2.5 <= high else 'NO'}"
)

`k` misses, exactly as the Laplace interval did in rung 4. It is the least constrained
parameter in the model, and a single dataset genuinely supports a value near 16.

That is worth being blunt about: **the posterior is a statement about this dataset, not
about the truth.** Refitting across many synthetic datasets shows the estimator is
essentially unbiased — this particular draw is simply one where the data pull low. A
posterior that covered the truth every time would be miscalibrated in the other direction.

## The ridge, as a posterior object

Rung 3 found the sloppy direction with a Jacobian. Here it appears directly as
correlation between draws.

In [ ]:
correlation = np.corrcoef(samples[:, 0], samples[:, 1])[0, 1]
print(f"corr(beta, gamma) = {correlation:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(samples[:, 0], samples[:, 1], s=3, alpha=0.15, color="#33628f")
axes[0].plot(truth["beta"], truth["gamma"], "*", color="#c1502e", ms=16, label="truth")
axes[0].set(xlabel="beta", ylabel="gamma", title=f"joint posterior (corr = {correlation:.3f})")
axes[0].legend()

axes[1].hist(r0, bins=50, color="#33628f", alpha=0.8)
axes[1].axvline(truth["beta"] / truth["gamma"], color="#c1502e", lw=2, label="truth")
axes[1].set(xlabel="R0", ylabel="draws", title="R0 = beta / gamma, with correlation intact")
axes[1].legend()
fig.tight_layout()

A correlation of about 0.99. The two rates are almost a single quantity as far as these
data are concerned — and `R0`, their ratio, is *narrower* than either marginal would
suggest, because the errors move together. Combining two separate intervals would have
given a wider and wrong answer.

## Against Laplace

Rung 4's cheap approximation, on the same data.

In [ ]:
objective = negative_log_likelihood(model, dataset.y, dataset.t)
mle = fit(objective, dataset.theta_true * 1.2, model.param_names)
laplace = laplace_intervals(mle.theta, observed_fisher(objective, mle.theta), model.param_names)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, name in zip(axes, model.param_names, strict=True):
    index = model.param_names.index(name)
    posterior_ci = np.percentile(samples[:, index], [2.5, 97.5])
    for row, (_label, (low, high)) in enumerate(
        (("Laplace", laplace[name]), ("posterior", tuple(posterior_ci)))
    ):
        ax.plot([low, high], [row, row], "-", lw=4, color="#33628f")
    ax.axvline(truth[name], ls="--", color="#c1502e", lw=1.5)
    ax.set(yticks=[0, 1], yticklabels=["Laplace", "posterior"], xlabel=name, ylim=(-0.6, 1.6))
    ax.grid(alpha=0.3, axis="x")
fig.suptitle("95% intervals: Laplace vs the posterior (dashed = truth)")
fig.tight_layout()

For `beta`, `gamma` and `rho` the two agree closely — which is the honest headline.
**On this problem Laplace was already good enough**, and rung 4 was not lying when it
said you could stop there. The disagreement is concentrated in `k`, the parameter whose
likelihood is least quadratic.

The reason to be here is not that the intervals moved. It is what comes next.

## What the posterior buys: a decision

Suppose an intervention cuts transmission by a fraction `c`. Whether it brings the
epidemic under control is the question of whether `(1 - c) * R0` falls below 1 — a
statement about `beta` and `gamma` *jointly*.

In [ ]:
reductions = np.linspace(0.0, 0.8, 33)
probability = [(r0 * (1 - c) < 1.0).mean() for c in reductions]

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(100 * reductions, probability, lw=2.5, color="#33628f")
ax.axhline(0.9, ls="--", color="#c1502e", label="90% confident")
ax.set(
    xlabel="reduction in transmission (%)",
    ylabel="P(epidemic under control)",
    ylim=(-0.02, 1.02),
)
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

needed = 100 * reductions[np.argmax(np.array(probability) >= 0.9)]
print(f"smallest reduction that is 90% likely to work: {needed:.0f}%")

That curve is the payoff. It is not an estimate with an error bar — it is a probability
attached to each candidate policy, obtained by pushing the joint posterior through a
function of the parameters. No interval from rung 4 could have produced it, because the
answer depends on `beta` and `gamma` together.

## What the black-box likelihood costs

The sampler above learns its proposal directions from the chain's own history. A
componentwise random walk — the textbook Metropolis — can only propose moves parallel to
the axes, and the probability mass here lies along a diagonal.

In [ ]:
naive = sample_posterior(
    model, dataset.y, dataset.t, draws=800, tune=800, seed=1, sampler="metropolis"
)
naive_report = check_convergence(naive, var_names=["beta", "gamma", "rho", "k"])

print(f"{'sampler':16} {'max R-hat':>10} {'min ESS':>9}   usable")
print(
    f"{'DEMetropolisZ':16} {report.max_r_hat:10.3f} {report.min_ess_bulk:9.0f}   {report.converged}"
)
print(
    f"{'Metropolis':16} {naive_report.max_r_hat:10.3f} {naive_report.min_ess_bulk:9.0f}   "
    f"{naive_report.converged}"
)

The naive sampler does not merely run slower — it fails outright. From 3200 draws it
retains an effective sample size in the *single digits*, with an R-hat far above any
usable threshold. The correlation diagnosed back at rung 3 is the
direct cause. Sloppiness is not only an inference nuisance; it breaks algorithms that do
not account for it.

This is also the clearest argument for gradients. NUTS would use the local geometry to
move along the ridge instead of fighting it. Getting there means a differentiable
implementation of the dynamics — which is a real engineering decision with a real cost,
not a free upgrade.

## When to stop here

**This is the top of the mechanistic ladder.** Priors are usable, uncertainty is joint,
and decisions can be stated as probabilities. For a model whose likelihood you can write
down and evaluate, there is nothing above this.

**What would push higher is not a better estimator — it is a different problem.**

- The *observation* model is a guess. `rho` constant, negative-binomial noise: real
  surveillance has weekday effects, reporting delays and testing regimes that change.
  Learning that link from data needs machinery this ladder does not have.
- The likelihood might not exist in closed form at all. Swap the deterministic ODE for a
  stochastic generator and there is no density left to evaluate — only the ability to
  simulate. That is what would force simulation-based
  inference: a different ladder, not a higher rung of this one.

Either extension would have to be measured against the posterior sampled here. For a
model whose likelihood you can write down and evaluate, it is the reference.